# From whole-slide images to tile embeddings with UNI2-h

**Goal.** Turn each whole-slide image (WSI) of the BRACS dataset into a set of fixed-size tile embeddings produced by the pathology foundation model UNI2-h, so that they can be used later for classification.

**Pipeline.**
1. **Setup**: library versions, compute device and imports.
2. **Data**: parameters, tissue filter, tile index, visual check and data loading.
3. **Feature extraction**: load UNI2-h and save one feature file per slide.
4. **Inspection**: check one feature file, then the embeddings of all slides.

**Inputs / outputs.** Input: `.svs` slides in `DATA_DIR`. Output: one `.pt` file per slide in `FEATURES_DIR/<model>/`, holding the CLS vector of every tile, the tile coordinates and the tiling parameters (and optionally the patch tokens).

**Reproducibility.** Library versions are printed in section 1.1. All tiling parameters are collected in section 2.1 and stored in every output file.

**Overview.** Flow of this notebook (sections, functions and files).

```mermaid
flowchart TD
    subgraph IN["Inputs"]
        SVS[("BRACS slides<br/>.svs files in DATA_DIR")]
        W[("UNI2-h weights<br/>original_vit_model/")]
    end

    subgraph S2["2. Data"]
        P["2.1 Parameters<br/>level, tile size, stride, batch size"]
        DS["2.2 WSITileDataset<br/>index tiles + tissue filter"]
        CHK["2.3 and 2.4 Checks<br/>tile count, plot_wsi"]
        DL["2.5 DataLoader<br/>ordered batches, worker processes"]
    end

    subgraph S3["3. Feature extraction"]
        M["3.1 Load UNI2-h<br/>+ model transform"]
        GF["3.2 generate_features<br/>CLS vectors, optional patch tokens"]
    end

    OUT[("FEATURES_DIR/UNI2-h/<br/>one .pt file per slide")]

    subgraph S4["4. Inspection"]
        I1["4.1 inspect_feature_file<br/>one slide"]
        I2["4.2 plot_cls_overview<br/>all slides"]
        I3["4.3 plot_cluster_quality<br/>choice of k"]
        I4["4.4 plot_class_analysis<br/>vs 7 classes + linear probe"]
    end

    SVS --> DS
    P --> DS
    DS --> CHK
    DS --> DL
    DL --> GF
    W --> M
    M --> GF
    GF --> OUT
    OUT --> I1
    OUT --> I2
    OUT --> I3
    OUT --> I4
    LAB[("labels/BRACS.xlsx<br/>class, patient, set")] --> I4
```

## 1. Setup

### 1.1 Environment and library versions
Prints the Python and library versions used in this run, so results can be traced back to the software that produced them.

In [1]:
# STEP 1: Import general-purpose third-party libraries (tensors, plotting, data handling)
import torch
import torchvision
import pandas as pd
import numpy as np
import sys

# STEP 2: Print library versions for reproducibility
print('System Version:', sys.version)
print('PyTorch version:', torch.__version__)
print('Torchvision version:', torchvision.__version__)
print('Numpy version:', np.__version__)
print('Pandas version:', pd.__version__)

System Version: 3.14.7 (main, Aug 10 2026, 07:46:56) [GCC 16.1.1 20260728]
PyTorch version: 2.15.0+cu132
Torchvision version: 0.29.2+cu132
Numpy version: 2.5.4
Pandas version: 3.1.1


### 1.2 Compute device
Selects the fastest available device (CUDA, then Apple MPS, then CPU) and enables cuDNN autotuning on GPU. This is safe here because every input tile has the same size.

In [2]:
# STEP 1: Select CUDA, then Apple MPS, then CPU. Pass e.g. get_device("cpu") to force one.
from utils.device import get_device

device = get_device()

# STEP 2: Enable cuDNN autotuning when on GPU (safe here since input tile size is fixed)
if device.type == "cuda":
    torch.backends.cudnn.benchmark = True
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
else:
    print(f"Using {device.type.upper()}")

# Expose the device to the notebook
device

Using CUDA: NVIDIA RTX PRO 4500 Blackwell


device(type='cuda')

### 1.3 Imports
Imports the project's own building blocks from `utils/`: dataset and tile filter, transforms, data loader, model loading, feature extraction and the inspection plots.

In [3]:
# STEP 1: Standard-library and plotting imports
from functools import partial
from pathlib import Path
from time import time
import matplotlib.pyplot as plt

# STEP 2: Import this project's own pipeline building blocks (dataset, filters,
# transforms, feature extraction, model loading, inspection plots)
from utils.dataset import WSITileDataset
from utils.clean_empty import background_filter
from utils.transformer_basic import build_basic_transform
from utils.dataloader import create_dataloader
from utils.transformer_per_model import build_uni2h_transform
from utils.load_model import load_model, MODEL_NAME
from utils.generate_features import generate_features
from utils.plot_wsi import plot_wsi
from utils.inspect_features import inspect_feature_file
from utils.cls_overview import plot_cls_overview
from utils.cluster_quality import plot_cluster_quality
from utils.class_analysis import plot_class_analysis

## 2. Data

### 2.1 Experiment parameters
All tiling settings in one place.
- `TILE_SIZE = 224` is the input size of UNI2-h. With `STRIDE == TILE_SIZE` the tiles do not overlap.
- `PYRAMID_LEVEL` selects the resolution: level 1 is full resolution, higher levels are more downsampled. The number of tiles grows roughly 16-fold for each level step towards level 1.
- `BATCH_SIZE` is the number of tiles per model call.

In [4]:
# Major variables
BATCH_SIZE = 16         # STEP 1: Tiles per batch fed through the model
# TODO: check input to the models if they come from here / single source of truth
TILE_SIZE = 448          # STEP 2: Width/height (in pixels) of each square tile
STRIDE = 448             # STEP 3: Distance between neighboring tile origins; STRIDE == TILE_SIZE -> no overlap
DATA_DIR='/mnt/Data/bracs/data'  # STEP 4: Directory of source WSI TIFFs
PYRAMID_LEVEL = 1        # STEP 5: Which pyramid resolution level to tile from (higher = more downsampled)
FEATURES_DIR = Path("/mnt/Scratch/phd_streamembed_features/level_1")
# TODO: think whether it makes sense to use the augmentation flags

### 2.2 Tile filter, transform and dataset index
- **Tile filter:** a tile is kept only if at least 50% of its pixels are neither near-white nor near-black, which removes background and empty glass.
- **Basic transform:** model-independent preprocessing; augmentations are switched off because this run only extracts features.
- **Dataset:** building it indexes every tile of every slide (in parallel) and applies the filter. A progress bar shows the slides indexed, the elapsed time and the estimated time left (a linear extrapolation over slides, so it is rough while the first slides are being processed; `show_progress=False` switches it off). Slides without the requested level, and tiles that cannot be read, are left out and reported as warnings.

In [ ]:
# STEP 1: Define the tissue/background filter — a tile is kept only if at least
# min_tissue_fraction of its pixels are neither near-white nor near-black
tile_filter = partial(
    background_filter,
    min_tissue_fraction=0.5,
    white_threshold=220,
    black_threshold=15,
)
print("Tile filter created")

# STEP 2: Define the shared, model-agnostic transform applied to every tile as it's
# loaded (to-tensor/dtype scaling here; augmentations disabled for this inspection run)
# TODO: Victor has suggested using a filter to have all slides with same H&E ton - DO THAT
basic_transform = build_basic_transform(
    apply_color_jitter=False,
    apply_horizontal_flip=False,
    apply_vertical_flip=False,
    apply_rotation=False,
)
print("Basic transform created")

# STEP 3: Index every tile across every slide in DATA_DIR at the target pyramid level,
# applying tile_filter during indexing so only tissue-containing tiles are kepts
dataset = WSITileDataset(
    data_dir=DATA_DIR,
    level=PYRAMID_LEVEL,
    tile_size=TILE_SIZE,
    stride=STRIDE,
    basic_transform=basic_transform,
    tile_filter=tile_filter,
)
print("Dataset created")


Tile filter created
Basic transform created


Indexing slides:   0%|          | 0/547 [00:00<?, ?slide/s]

Exception ignored from cffi callback <function _log_handler_callback at 0x7f9929331bc0>:
Traceback (most recent call last):
  File "/home/guisazevedo/Projects/01_accamargo/phd_streamembed/.venv/lib/python3.14/site-packages/pyvips/__init__.py", line 146, in _log_handler_callback
Exception ignored from cffi callback <function _log_handler_callback at 0x7f9929335bc0>:
Traceback (most recent call last):
  File "/home/guisazevedo/Projects/01_accamargo/phd_streamembed/.venv/lib/python3.14/site-packages/pyvips/__init__.py", line 146, in _log_handler_callback
Exception ignored from cffi callback <function _log_handler_callback at 0x7f9929331bc0>:
Traceback (most recent call last):
  File "/home/guisazevedo/Projects/01_accamargo/phd_streamembed/.venv/lib/python3.14/site-packages/pyvips/__init__.py", line 146, in _log_handler_callback
Exception ignored from cffi callback <function _log_handler_callback at 0x7f9929331bc0>:
Traceback (most recent call last):
  File "/home/guisazevedo/Projects/01_a

### 2.3 Dataset size
Number of tiles that passed the filter and number of slides skipped because they lack the requested pyramid level. Check this against the expected cohort size before the long run.

In [ ]:
print(len(dataset), "tiles;", len(dataset.skipped_slides), "slides skipped")

### 2.4 Visual check of the tiling
For each chosen slide, `plot_wsi` shows (top) the slide at the tiled level, (middle) the tile grid with the tiles that passed the filter outlined in blue and the first 12 outlined in orange, and (bottom) those 12 tile crops as the dataset returns them.

Check that tissue is kept, glass is dropped and the crops look like what the model should see. Large pyramid levels take several seconds per slide to draw.

In [ ]:
# STEP 1: Choose the slides to draw (position among the slides that have tiles, or a file name)
SLIDES_TO_PLOT = [0, 1, 2]

# STEP 2: Draw each slide with its tile grid and its first tile crops
for slide in SLIDES_TO_PLOT:
    _ = plot_wsi(dataset, slide=slide, n_tiles=12)

### 2.5 Data loader
Batches the tiles for inference. `shuffle=False` is required: features are written slide by slide, so the tiles of one slide must arrive together and in order. Several worker processes read tiles while the model runs.

In [ ]:
# STEP 1: Build the DataLoader that batches tiles for model inference below
dataloader = create_dataloader(
    dataset = dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    device=device,
    num_workers=32
)

## 3. Feature extraction

### 3.1 Load the model
Builds the UNI2-h architecture, loads the local checkpoint from `original_vit_model/` and sets it to evaluation mode. The matching model-specific transform (resize and ImageNet normalization) is created alongside.

In [ ]:
# STEP 1: Load the UNI2-h model (weights in original_vit_model/) and its transform
transform = build_uni2h_transform()
model = load_model(device=device)

### 3.2 Extract and save features
Runs every tile through UNI2-h and writes one `.pt` file per slide to `FEATURES_DIR/<model>/<slide>.pt`, containing `cls_features`, `coordinates`, `model_encoder`, `tile_params` and, only if `save_patch_features=True`, `patch_features`.

CLS-only files are about 250 times smaller than files with patch tokens. This is the long-running step; the printed timing allows extrapolating to other pyramid levels.

In [ ]:
# STEP 1: Directory where every model's features will be written, one subfolder per model
from pathlib import Path

# STEP 2: Run all tiles through the model and save the resulting
# per-slide feature files (see utils/generate_features.py for the extraction
# + saving logic)
start = time()
features = generate_features(
    dataloader=dataloader,
    transform=transform,
    model=model,
    device=device,
    max_tiles_to_print=1,
    output_dir=FEATURES_DIR,
    model_encoder=MODEL_NAME,
    save_patch_features=False,
)
end = time()

# STEP 3: Report timing for this run
print(f"Time taken: {end - start:.3f} seconds")
print(f"Time per tile: {(end - start) / len(dataset):.3f} seconds")
print(f"Pyramid level: {PYRAMID_LEVEL}")
# TODO: dont need to print "saved features"

## 4. Inspection of the features

### 4.1 One feature file
Verifies a single feature file before further analysis. The text output lists the path, size, shapes, data types, a finite-value check, summary statistics of the CLS embeddings and the stored tiling parameters (level, tile size, stride, filter). The figure uses the CLS embeddings only; patch tokens, if the file contains them, are not plotted. Each panel carries its own explanation; the main points are summarised here.

**Slide atlas.** Every tile is coloured by the first three principal components of its CLS embedding, so that tiles with similar colours have similar embeddings. A coherent spatial pattern that follows tissue morphology indicates that the embedding encodes tissue structure; noise unrelated to the layout indicates that it does not. The principal components are fitted on this slide, so colours cannot be compared between slides.

**CLS norm.** The norm of a tile is the Euclidean length of its CLS vector $h$, $\lVert h\rVert_2=\sqrt{\sum_d h_d^2}$ over the embedding dimensions: one number per tile that measures the size of the vector and ignores its direction. The histogram shows how it is distributed over the tiles of the slide. The norm is not a quality score, but isolated extreme values usually correspond to artefacts (pen marks, folds, blur, glass) and should be inspected before such tiles enter later analyses.

**Spread of the dimensions.** The per-dimension standard deviation across the tiles, sorted in decreasing order, shows whether between-tile variation is concentrated in a few dimensions or spread over many. Individual dimensions are not interpretable. For classification, the relevant variation lies between slides and classes, which Sections 4.2 and 4.4 address.


In [ ]:
# STEP 1: Summarize and plot one file. Pass feature_file="<path>.pt" to pick a specific slide;
# by default the first file of FEATURES_DIR / MODEL_NAME is used
_ = inspect_feature_file(
    features_dir=FEATURES_DIR,
    model_name=MODEL_NAME,
    dataset=dataset,   # only used for the tiling parameters of files that do not store them
    n_show=8,
)

### 4.2 CLS embeddings of all slides
Describes the structure of the embedding space across slides, using only the CLS embeddings, which are the input of the later classifier. PCA and k-means are fitted on a random subset of tiles and then applied to all tiles. The clusters are unsupervised groups of similar tile appearance; they are not diagnostic categories. Each panel carries its own explanation.

**Structure of the space (first row).** The density plot shows whether tiles form separate groups or a continuum in the plane of the first two principal components. The cluster plot colours the same plane by k-means cluster; overlap in two dimensions does not exclude separability in the remaining components. The variance panel gives the fraction of variance explained per component: if few components reach 90%, the two-dimensional views are representative; otherwise they show only a small part of the information. A dominant first component may reflect a slide-level effect (staining, scanner) rather than tissue.

**Meaning of the clusters (second row).** The cluster maps show whether clusters follow recognisable morphological regions consistently across slides; scattered assignments, or clusters confined to one slide, point to noise or slide-specific appearance. The cluster-mix bars give the composition of each slide. Differences in composition are useful for classification only if they follow the diagnosis, not the scanner, site or staining batch.

**Implications for classification (third row).** The slide-effect panel compares the cosine similarity of tile pairs from the same slide with pairs from different slides. Higher similarity within slides indicates a slide-specific component in the embedding, so training, validation and test sets must be split by slide or patient, never by tile; otherwise a classifier can reach high accuracy by recognising the slide. The distribution of tiles per slide shows how unevenly slides contribute, which matters for any tile-level statistic and loss.

In [ ]:
# STEP 1: Summarize the CLS embeddings of every slide in FEATURES_DIR / MODEL_NAME.
# max_slides=None uses all slides; n_clusters can be any number >= 2 (colours beyond 3 are not colour-blind-safe)
overview = plot_cls_overview(
    features_dir=FEATURES_DIR,
    model_name=MODEL_NAME,
    max_slides=None,
    n_clusters=5,
)

### 4.3 Choosing the number of clusters
The number of clusters k in Section 4.2 is a modelling choice and needs a justification. This section repeats k-means for a range of k on different random subsamples of the tiles and scores each partition on the same held-out tiles. Each panel carries its own explanation.

- **Inertia (elbow)** is the mean squared distance of a tile to its cluster centre. It decreases monotonically with k, so only a bend in the curve is informative.
- **Silhouette** (Rousseeuw, 1987) compares, for each tile, the distance to its own cluster with the distance to the nearest other cluster. Values up to 0.25 indicate no substantial structure, 0.26 to 0.50 weak, 0.51 to 0.70 reasonable and above 0.70 strong structure (Kaufman and Rousseeuw, 1990).
- **Davies-Bouldin index** (Davies and Bouldin, 1979) relates cluster spread to the distance between cluster centres; lower is better. Agreement with the silhouette supports a choice of k.
- **Stability** is the adjusted Rand index (Hubert and Arabie, 1985) between partitions obtained from different subsamples. Low values mean that the partition depends on the sample and should not be interpreted.

If no k stands out on these criteria, k should be set by the research question (for example k = 7 for the seven BRACS classes) and reported as a modelling choice, not as a property of the data.

In [ ]:
# STEP 1: Compare k = 2..12 (5 repeats each). Lower n_repeats or k_values for a quicker run.
quality = plot_cluster_quality(
    features_dir=FEATURES_DIR,
    model_name=MODEL_NAME,
    k_values=range(2, 13),
    n_repeats=5,
)

### 4.4 Embeddings and class labels
Relates the CLS embeddings to the seven BRACS classes (N, PB, UDH, FEA, ADH, DCIS, IC), read from `labels/BRACS.xlsx`. Each panel carries its own explanation.

**Weak labels.** Each tile inherits the label of its slide, but a slide labelled invasive carcinoma also contains normal tissue. Class separation at tile level is therefore expected to be modest, and the slide level is the appropriate test.

**Unsupervised view.** The projection coloured by class and the class-balance chart show whether classes occupy different regions and how many slides each class has; with imbalance, balanced accuracy and per-class results should be reported. The contingency table between classes and k-means clusters, summarised by normalised mutual information (NMI) and the adjusted Rand index (ARI), shows how far the unsupervised structure follows the labels. The per-class silhouette quantifies how well each class label separates tiles.

**Supervised baseline (printed, not plotted).** A class-balanced logistic regression on the mean CLS embedding of each slide is evaluated with cross-validation in which all slides of a patient fall in the same fold, which avoids leakage between related slides. Its accuracy, balanced accuracy (with a 95% bootstrap interval over slides) and macro F1 are printed in the text output, together with the baselines they should be read against: chance, the majority class, the same pipeline trained on shuffled labels (a result clearly above chance would indicate leakage) and the official Training to Testing split. The probe is untuned and uses mean-pooled slide vectors, so it is a lower bound for the classifier of `02_vectors_to_classifier.ipynb`; the same numbers are in the dictionary returned by `plot_class_analysis`.


In [ ]:
# STEP 1: Path to the BRACS label sheet (WSI label, patient id and official set per slide)
LABELS_FILE = Path("labels/BRACS.xlsx")

# STEP 2: Compare the embeddings with the seven classes. n_clusters sets the number of k-means
# clusters compared with the classes; slides without a label are dropped with a warning
class_analysis = plot_class_analysis(
    features_dir=FEATURES_DIR,
    model_name=MODEL_NAME,
    labels_file=LABELS_FILE,
    n_clusters=5,
)